# Sesión 5 — MUBD

**Clustering: descubrir grupos de observaciones parecidas**

**Pregunta inicial:** ¿qué grupos aparecen al comparar las medidas sin utilizar las clases conocidas?

Trabajaremos con Iris, vinos y clientes de un mayorista. Veremos cómo cambian los grupos con la escala, la distancia, el método y el número de grupos.

**Recorrido:** escala → clustering jerárquico → Hopkins → vinos → k-means → disco y corona → clientes → conclusiones.

**Cómo trabajar:** ejecuta los ejemplos en orden y consulta las secciones indicadas en cada actividad. En los ejercicios 5-1 y 5-2 solo está preparada la carga; escribe tú las soluciones. Los ejemplos posteriores a vinos no dependen de completar ese ejercicio.

**Clustering no es clasificación:** aquí descubrimos grupos sin usar etiquetas. Clasificar sería aprender de especies conocidas para predecir la especie de una flor nueva. Las especies y tipos de vino se consultan al final; un grupo descubierto no tiene por qué ser una clase conocida.

Ejecuta con **Shift + Enter**. Los controles permiten explorar otras opciones. Para empezar de nuevo: **Reiniciar kernel → Ejecutar todo**.

## 5.0. Preparación

Selecciona el kernel del proyecto y ejecuta los ejemplos en orden. No necesitas reinstalar las bibliotecas en cada sesión. Las semillas permiten repetir los ejemplos: **12345**, salvo la comparación que indica **1234**.

Las funciones de apoyo contienen el código largo. Primero entiende qué datos reciben, qué devuelven y cómo se utilizan.

**Conceptos que trabajaremos:** distancias y jerarquía, tendencia al agrupamiento con Hopkins, k-means, indicadores para comparar particiones y combinaciones de métodos.

**Cómo comparar los ejemplos:** conservamos Iris original como referencia y estudiamos la estandarización aparte. Las inercias calculadas con medidas originales y estandarizadas no son directamente comparables. Las semillas y los centros iniciales también pueden cambiar el resultado. Los ejercicios de vinos y mayorista permiten practicar estos conceptos con otros conjuntos de datos.

Los índices, ARI y el kernel ayudan a comparar resultados. Al final se incluye una lectura opcional sobre DBSCAN, sin añadir otra práctica.

In [ ]:
# Path permite construir rutas de archivos sin depender del sistema operativo.
from pathlib import Path
# sys permite añadir la carpeta de scripts a la búsqueda de módulos de Python.
import sys

# Busca la carpeta del proyecto empezando en la carpeta actual y subiendo hacia sus padres.
PROYECTO = next((carpeta for carpeta in (Path.cwd(), *Path.cwd().parents)
                 # Reconoce la raíz del proyecto porque allí existe este archivo de datos.
                 if (carpeta / 'datos' / 'wine2.txt').is_file()), None)
# Detiene la ejecución con una indicación útil si el cuaderno se abrió fuera del proyecto.
if PROYECTO is None:
    raise FileNotFoundError('Abre el cuaderno dentro del proyecto LaSalle_Estadistica.')
# Guarda la carpeta con los módulos auxiliares de la sesión 5.
SCRIPTS_PATH = PROYECTO / 'sesiones' / 'sesion5'
# Añade la carpeta solo si todavía no está en la lista de búsqueda de Python.
if str(SCRIPTS_PATH) not in sys.path:
    # Así se pueden importar los scripts al ejecutar desde la raíz o desde la sesión.
    sys.path.insert(0, str(SCRIPTS_PATH))
# Guarda la carpeta que contiene los archivos de datos.
DATA_PATH = PROYECTO / 'datos'

In [ ]:
# Si faltan bibliotecas, quita el símbolo # al inicio y ejecuta esta línea una sola vez.
# %pip install numpy pandas matplotlib scipy scikit-learn seaborn ipywidgets threadpoolctl ipykernel

In [ ]:
# partial permite fijar argumentos de funciones para construir callbacks reutilizables.
from functools import partial
# SimpleNamespace agrupa resultados relacionados en un objeto sencillo.
from types import SimpleNamespace
# NumPy ofrece arrays y operaciones numéricas.
import numpy as np
# pandas organiza los datos en tablas con filas y columnas.
import pandas as pd
# pyplot crea y muestra gráficos.
import matplotlib.pyplot as plt
# seaborn añade estilos y gráficos estadísticos a Matplotlib.
import seaborn as sns
# ipywidgets crea controles interactivos en el cuaderno.
import ipywidgets as widgets
# display muestra tablas, gráficos y controles en la salida de una celda.
from IPython.display import display
# Estas figuras geométricas sirven para dibujar zonas y marcas en los gráficos.
from matplotlib.patches import Circle, Rectangle, Wedge
# scatter_matrix dibuja gráficos de dispersión para cada pareja de columnas.
from pandas.plotting import scatter_matrix
# stats reúne pruebas y distribuciones estadísticas de SciPy.
from scipy import stats
# Estas funciones calculan, cortan y dibujan árboles de clustering jerárquico.
from scipy.cluster.hierarchy import linkage, cophenet, cut_tree, dendrogram
# eigh calcula autovalores y autovectores de matrices simétricas.
from scipy.linalg import eigh
# Estas herramientas calculan distancias entre filas y las convierten a matriz.
from scipy.spatial.distance import pdist, squareform, cdist
# linear_sum_assignment encuentra el emparejamiento de etiquetas con mayor solapamiento.
from scipy.optimize import linear_sum_assignment
# KMeans proporciona el algoritmo de Lloyd de scikit-learn.
from sklearn.cluster import KMeans
# load_iris carga el conjunto de datos didáctico de flores Iris.
from sklearn.datasets import load_iris
# PCA proyecta variables numéricas en componentes principales.
from sklearn.decomposition import PCA
# Estas métricas comparan la calidad interna de una partición o su acuerdo externo.
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score,
)
# threadpool_limits limita los hilos numéricos para mantener los ajustes reproducibles.
from threadpoolctl import threadpool_limits
# Importa funciones que ordenan controles y los presentan en paneles.
from s5_controles import cabecera_widget, fila_controles, panel_interactivo
# Importa gráficos, estilos, colores y herramientas para mostrar imágenes dentro de widgets.
from s5_graficos import (
    configurar_estilo, estilo_ejes, mostrar_html, imagen_control,
    COLOR_BASE, COLOR_ACENTO, COLORES_GRUPO, MAPA_GRUPOS,
)
# Indica a Matplotlib que dibuje los gráficos dentro del cuaderno.
%matplotlib inline
# Hace que la celda muestre todos los resultados de expresiones, no solo el último.
%config InteractiveShell.ast_node_interactivity = "all"
# Aplica el estilo común del proyecto a las figuras.
configurar_estilo()
# Fija la semilla usada en los ejemplos para que las muestras aleatorias se repitan.
SEED = 12345

### 5.0.1. Funciones para escalar, resumir y ajustar

- `escalar`: resta la media y divide por la desviación típica **muestral** de cada columna (`ddof=1`). Recibe medidas numéricas, sin ausentes ni columnas constantes. `StandardScaler` usa otra convención (`ddof=0`): los valores estandarizados no son exactamente iguales.
- `resumen_particion`: recibe los datos utilizados para agrupar y sus etiquetas. Calcula centros, tamaños y dispersión.
- `ajustar_kmeans`: permite elegir Lloyd o la implementación didáctica de transferencias Hartigan–Wong del proyecto. **No estandariza automáticamente.** `n_init` cuenta puntos de partida y conserva el de menor WSS. `init` puede recibir una matriz con un centro por fila; en ese caso se hace un único inicio.
- `renumerar`: cambia los nombres de los grupos, no sus miembros.

Importar estas funciones todavía no agrupa Iris. La implementación del proyecto no garantiza los mismos resultados que `kmeans` de R.

In [ ]:
# Importa funciones para estandarizar datos, renombrar grupos, resumir particiones y ajustar k-means.
from s5_modelos import escalar, renumerar, resumen_particion, ajustar_kmeans
# La importación solo carga definiciones; aún no agrupa ninguna flor.

### 5.0.2. Herramientas de representación

`pares_coloreados` dibuja pares de medidas. `representar_clusters` muestra grupos ya calculados en dos componentes principales: dos ejes que resumen las medidas.

Por defecto, `representar_clusters` estandariza **para dibujar**, sin cambiar los grupos. Con `estandarizar=False`, utiliza la escala recibida.

Los porcentajes de los ejes indican cuánta variación conserva el dibujo. Los contornos ayudan a ver la zona ocupada por cada grupo; no son fronteras exactas.

In [ ]:
# Importa funciones para representar datos y colorear pares de medidas con etiquetas dadas.
from s5_graficos import representar_clusters, pares_coloreados
# Estas funciones dibujan etiquetas existentes; no calculan ni cambian los grupos.

## 5.1. Inspeccionar Iris y justificar la escala

Iris contiene **150 flores** y cuatro medidas en centímetros: longitud y anchura de sépalo y pétalo.

Guardamos la especie en `Species` para compararla al final. Para agrupar utilizamos solo las cuatro medidas de `iris2`.

Antes de calcular distancias, revisamos cuánto varía cada medida. Una variable con diferencias mayores puede pesar más, aunque todas estén en centímetros.

Los ejemplos principales usan las medidas originales. `iris2_escalado` guarda una versión estandarizada para estudiar cómo cambia la agrupación.

In [ ]:
# Carga las medidas y las etiquetas de especie del conjunto Iris.
dataset = load_iris()
# Construye una tabla con las cuatro medidas y nombres de columnas más claros.
iris = pd.DataFrame(dataset.data, columns=["Sepal.Length", "Sepal.Width", "Petal.Length", "Petal.Width"])
# Numera las flores desde 1 para facilitar su identificación en los ejemplos.
iris.index = np.arange(1, len(iris) + 1)
# Convierte los códigos numéricos de especie en nombres; se reserva para comparar después.
iris["Species"] = pd.Categorical.from_codes(dataset.target, dataset.target_names)
# Muestra la tabla con medidas y especie conocida.

In [ ]:
# Copia solo las cuatro columnas de medidas; la especie no se usará para agrupar.
iris2 = iris.iloc[:, :4].copy()
# Muestra los datos numéricos que entrarán en las distancias.
iris2
# Resume cantidad, media, desviación típica, mínimo y máximo de cada medida.
display(iris2.describe().T[["count", "mean", "std", "min", "max"]].round(3))
# Comprueba que hay 150 flores, cuatro medidas y ningún dato ausente.

### 5.1.1. Leer el resumen y los gráficos de pares

Cada medida tiene **150 valores**. Sus desviaciones típicas son aproximadamente **0,828; 0,436; 1,765 y 0,762 cm**, siguiendo el orden de la tabla.

`Petal.Length` varía más que `Sepal.Width`, por lo que puede influir más en las distancias sin estandarizar. `describe()` muestra media (`mean`), mediana (`50%`), desviación típica (`std`) y máximo (`max`). `head()` permite ver solo las primeras filas.

Para revisar otra tabla: `shape` da filas y columnas; `dtypes`, los tipos de columnas; `isna().sum()`, los ausentes de cada columna. Antes de agrupar necesitamos medidas numéricas y sin ausentes.

En la matriz de gráficos, la **columna** indica el eje horizontal y la **fila** el vertical. La diagonal muestra la distribución de cada medida. Busca relaciones y separaciones entre nubes, todavía sin utilizar las especies.

In [ ]:
# Dibuja histogramas en la diagonal y relaciones entre cada pareja de medidas.
_ = scatter_matrix(iris2, figsize=(9, 9), diagonal="hist", s=15, alpha=0.6)
# Presenta la matriz de gráficos en el cuaderno.

### 5.1.2. Por qué importa estandarizar

Comparamos las mismas **50 flores**, con distancia euclídea y enlace completo. Solo cambia la escala.

Estandarizar consiste en calcular:

$$z_{ij}=\frac{x_{ij}-\bar{x}_j}{s_j}.$$

Al valor de cada medida le restamos su media y lo dividimos por su desviación típica muestral. En las **150 flores**, cada columna queda con media 0 y desviación típica 1; después seleccionamos las 50 del ejemplo.

Así una medida no pesa más solo porque varía más. Esto no decide su importancia científica.

Cortamos ambos árboles en **tres grupos** y contamos los cambios de pertenencia. Antes emparejamos los números de grupo, porque sus nombres son arbitrarios. Las alturas de los árboles están en escalas distintas.

In [ ]:
# Estandariza cada una de las cuatro medidas usando su media y desviación típica muestral.
iris2_escalado = escalar(iris2)
# Selecciona 50 flores al azar de forma reproducible para comparar escalas.
muestra = iris2.sample(n=50, random_state=SEED)
# Toma las versiones estandarizadas de exactamente las mismas flores.
muestra_escalada = iris2_escalado.loc[muestra.index]

# Calcula las distancias euclídeas y construye árboles de enlace completo en escala original.
arbol_original = linkage(pdist(muestra), method="complete")
# Repite el cálculo con las medidas estandarizadas.
arbol_escalado = linkage(pdist(muestra_escalada), method="complete")
# Corta el primer árbol para asignar cada flor a uno de tres grupos.
grupos_original = cut_tree(arbol_original, n_clusters=3).ravel()
# Corta el segundo árbol también en tres grupos.
grupos_escalado = cut_tree(arbol_escalado, n_clusters=3).ravel()
# Cuenta cuántas flores de cada grupo original aparecen en cada grupo escalado.
tabla_cambios = pd.crosstab(grupos_original, grupos_escalado)

# Encuentra qué nombres de grupo se deben emparejar para maximizar coincidencias.
filas, columnas = linear_sum_assignment(-tabla_cambios.to_numpy())
# Construye la correspondencia entre etiquetas, que son números arbitrarios.
alineacion = {columnas[j]: filas[j] for j in range(len(filas))}
# Renombra las etiquetas estandarizadas para compararlas con las originales.
grupos_escalado_alineados = np.array([alineacion[g] for g in grupos_escalado])
# Marca las flores cuya pertenencia cambió tras alinear los nombres de grupo.
cambiaron = grupos_original != grupos_escalado_alineados

# Prepara dos paneles para comparar visualmente los árboles.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# Dibuja el árbol sin estandarizar y etiqueta sus hojas con el identificador de flor.
_ = dendrogram(arbol_original, labels=muestra.index.astype(str), leaf_rotation=90, ax=axes[0])
# Dibuja el árbol estandarizado usando los mismos identificadores.
_ = dendrogram(arbol_escalado, labels=muestra.index.astype(str), leaf_rotation=90, ax=axes[1])
# Añade título y nombres de ejes al panel de datos originales.
_ = axes[0].set(title="Sin normalizar", xlabel="Flor", ylabel="Distancia (cm)")
# Añade título y nombres de ejes al panel de datos estandarizados.
_ = axes[1].set(title="Normalizado", xlabel="Flor", ylabel="Distancia")
# Coloca un título común encima de ambos paneles.
_ = fig.suptitle("Mismas 50 flores, con y sin normalización")
# Ajusta espacios para que títulos y etiquetas no se solapen.
fig.tight_layout()
# Muestra los dos dendrogramas.
plt.show()

# Informa cuántas flores cambiaron de grupo entre las dos escalas.
print(f"Con k=3, cambian de grupo {cambiaron.sum()} de las {len(muestra)} flores.")
# Muestra algunos identificadores de las flores que cambiaron.
print("Algunos identificadores que cambian:", muestra.index[cambiaron][:10].tolist())
# Nombra la dimensión de filas de la tabla cruzada.
tabla_cambios.index.name = "Grupo sin normalizar"
# Nombra la dimensión de columnas de la tabla cruzada.
tabla_cambios.columns.name = "Grupo normalizado"
# Presenta los conteos de coincidencias entre particiones.

**Resultado:** al estandarizar, **9 de las 50 flores cambian de grupo**; entre ellas, las flores **57, 86 y 71**. La tabla cuenta los miembros compartidos entre ambas versiones.

La escala puede cambiar qué flores consideramos próximas.

#### Micropráctica 1 (opcional) — comprobar la escala

**Consulta:** `escalar` en **5.0.1** y la estandarización en **5.1.2**.

Completa el nombre de la columna de longitud del pétalo. Compara su desviación típica en `iris2` y `iris2_escalado` usando `std(ddof=1)`.

Antes de ejecutar, predice el resultado estandarizado e indica las unidades de ambos resultados.

In [ ]:
# Define el nombre de la medida de longitud del pétalo que pide la práctica.
variable_practica = "Petal.Length"
# Calcula la desviación típica en centímetros y después de estandarizar, sin unidades.
desviaciones_practica = pd.Series({
    "Desviación original (cm)": iris2[variable_practica].std(ddof=1),
    "Desviación estandarizada (sin unidades)": iris2_escalado[variable_practica].std(ddof=1),
})
# Muestra los dos resultados para compararlos.
display(desviaciones_practica)

## 5.2. Clustering jerárquico

Aquí usamos el método **aglomerativo**: empezamos con una flor por grupo, unimos dos grupos en cada paso y actualizamos sus distancias hasta obtener un árbol. Después lo cortamos para formar $k$ grupos. No hay que fijar $k$ para construirlo.

El método **divisivo** hace lo contrario: empieza con todas las observaciones y divide grupos. AGNES es un método aglomerativo y DIANA, uno divisivo; `linkage` construye la jerarquía aglomerativa, no DIANA.

Elegimos dos cosas distintas:
- **Distancia entre flores:** euclídea mide separación en línea recta; Manhattan suma diferencias absolutas.
- **Enlace entre grupos:** simple usa la pareja más cercana; completo, la más alejada; promedio, la distancia media entre sus miembros; Ward busca el menor aumento de dispersión interna.

Para dos filas $x$ e $y$, con $p$ medidas:

$$d_E(x,y)=\sqrt{\sum_{j=1}^{p}(x_j-y_j)^2},\qquad d_M(x,y)=\sum_{j=1}^{p}|x_j-y_j|.$$

Por ejemplo, entre $(0,0)$ y $(3,4)$, la distancia euclídea es 5 y Manhattan es 7. **Distancia euclídea no es distancia al cuadrado**: el cuadrado se utiliza en WSS y Ward.

Existen otras distancias: máxima usa la mayor diferencia; Mahalanobis considera la variación y relación entre medidas; Canberra compara diferencias relativas; Gower permite combinar tipos de variables; Jaccard compara presencias compartidas. No las calculamos aquí. Para Jaccard, la fracción de presencias compartidas es una **similitud**; la distancia es uno menos esa similitud.

### 5.2.1. Matriz de distancias

`pdist(iris2)` calcula las **11 175 parejas** de flores, sin repetirlas. `squareform` organiza esas distancias en una matriz de **150 × 150**.

Cada fila y columna identifica una flor. La diagonal es cero y la matriz es simétrica. Usamos distancia euclídea con las cuatro medidas originales. El número de parejas crece como $n(n-1)/2$: con muchos datos, guardar todas las distancias puede resultar costoso.

In [ ]:
# Calcula una distancia euclídea por cada pareja distinta de flores, en centímetros.
d = pdist(iris2, metric="euclidean")
# Convierte el vector de distancias en una tabla cuadrada con identificadores de flor.
matriz_distancias = pd.DataFrame(squareform(d), index=iris2.index, columns=iris2.index)
# Muestra la matriz para comparar visualmente distancias entre filas y columnas.
matriz_distancias
# Imprime cantidad de parejas y distancias extrema, con tres decimales.

**Resultado:** la distancia máxima es **7,085 cm**. Algunas parejas tienen distancia cero porque sus cuatro medidas registradas coinciden; eso no obliga a eliminar esas flores.

### 5.2.2. Jerarquización con enlace completo

`linkage(d, method="complete")` construye el árbol utilizando la mayor distancia entre una flor de cada grupo.

`hc` registra las **149 uniones**: qué ramas se unen, a qué altura y cuántas flores reúnen. Todavía no hemos elegido el número de grupos.

In [ ]:
# Construye un árbol jerárquico aglomerativo con enlace completo usando las distancias calculadas.
hc = linkage(d, method="complete")
# Se conserva este árbol como referencia; las exploraciones crearán árboles separados.

### 5.2.3. Leer el dendrograma

Cada hoja representa una flor. Dos ramas se unen a una altura que indica su separación según el enlace utilizado. Aquí la altura se lee en el **eje vertical**; en un dendrograma girado se leería en el horizontal. La orientación no cambia las uniones.

Con **enlace completo**, la altura es la mayor distancia entre los miembros de los dos grupos unidos. Una unión baja indica mayor proximidad; la altura no indica cuántas flores hay.

**Ward** utiliza el aumento de dispersión interna al unir dos grupos:

$$\Delta W=\frac{n_A n_B}{n_A+n_B}\lVert\bar{x}_A-\bar{x}_B\rVert^2.$$

$n_A$ y $n_B$ son sus tamaños; $\bar{x}_A$ y $\bar{x}_B$, sus centros. Ward elige la unión con menor aumento.

**Ward requiere distancia euclídea.** SciPy representa su altura como $\sqrt{2\Delta W}$, no directamente como $\Delta W$. Por eso no se lee igual que la del enlace completo. `dendrogram` dibuja el árbol.

In [ ]:
# Importa una función auxiliar que dibuja el dendrograma con identificadores de flor.
from s5_graficos import dibujar_dendrograma

# Dibuja el árbol jerárquico completo usando las etiquetas de las flores.

**Resultado del árbol fijo:** la última unión está a **7,085 cm**, la mayor distancia entre flores. Los colores no representan especies.

#### Explorar escala, distancia y enlace

Cambia una opción cada vez y observa el árbol.

La **distancia cofenética** de dos flores es la altura a la que aparecen juntas por primera vez. La comparamos con su distancia de entrada mediante una correlación de Pearson:

$$r_{\mathrm{cof}}=\frac{\sum_{i<j}(d_{ij}-\bar d)(c_{ij}-\bar c)}{\sqrt{\sum_{i<j}(d_{ij}-\bar d)^2\;\sum_{i<j}(c_{ij}-\bar c)^2}}.$$

$d$ contiene las distancias de entrada y $c$ las alturas de unión; las barras indican sus medias. La fórmula resume cómo varían juntas ambas listas. Cuanto más cerca de 1, mejor refleja el árbol su relación lineal; no exige distancias iguales ni elige $k$.

Cada árbol se compara con sus propias distancias de entrada. El control no modifica `hc` ni `ct`, y evita combinar Ward con Manhattan.

**Sin controles:** utiliza `display(widgets.HTML(value=explorar_arbol("euclidean", "complete", False)))`.

In [ ]:
# Define una función para comparar árboles con distancia, enlace y escala elegidos.
def explorar_arbol(distancia="euclidean", enlace="complete", estandarizar=False):
    # Evita usar Ward con una distancia que no sea euclídea.
    if enlace == "ward" and distancia != "euclidean":
        # Explica la combinación no válida para interpretar aumentos de inercia.
        raise ValueError("Para interpretar Ward como incremento de inercia usa distancia euclídea.")
    # Estandariza Iris solo si el usuario activó esa opción.
    tabla = escalar(iris2) if estandarizar else iris2
    # Calcula las distancias entre las flores con la métrica elegida.
    disimilitudes = pdist(tabla, metric=distancia)
    # Construye el árbol jerárquico con el método de enlace seleccionado.
    arbol = linkage(disimilitudes, method=enlace)
    # Calcula cuánto se correlacionan las distancias originales y las alturas del árbol.
    calidad = cophenet(arbol, disimilitudes)[0]
    # Crea una figura y un eje para dibujar el dendrograma.
    fig, ax = plt.subplots(figsize=(12, 4))
    # Dibuja el árbol sin etiquetas de hojas para que la figura sea legible.
    _ = dendrogram(arbol, no_labels=True, ax=ax, color_threshold=0,
                   above_threshold_color=COLOR_BASE)
    # Título y ejes indican método, distancia, escala y altura de fusión.
    _ = ax.set(title=f"{enlace.title()} · {distancia} · "
                     f"{'estandarizado' if estandarizar else 'sin escalar'}",
               xlabel="Flores ordenadas por el árbol", ylabel="Altura de la fusión")
    # Ajusta los márgenes para que el título y las etiquetas quepan.
    fig.tight_layout()
    # Devuelve el dendrograma en HTML junto con su correlación cofenética.
    return imagen_control(fig, "Dendrograma comparativo de Iris") + (
        f"<p>Correlación cofenética con su propia distancia: <b>{calidad:.3f}</b>. "
        "Un valor alto no demuestra que exista un número verdadero de grupos.</p>")


# Crea un menú para seleccionar distancia euclídea o Manhattan.
selector_distancia = widgets.Dropdown(options=[("Euclídea", "euclidean"),
                                               ("Manhattan", "cityblock")],
                                     value="euclidean", description="Distancia")
# Crea un menú para elegir el enlace del árbol jerárquico.
selector_enlace = widgets.Dropdown(options=[("Completo", "complete"),
                                            ("Promedio", "average"),
                                            ("Simple", "single"), ("Ward", "ward")],
                                  value="complete", description="Enlace")
# Añade una casilla para decidir si las medidas se estandarizan.
selector_escala = widgets.Checkbox(value=False, description="Estandarizar")
# Reserva un espacio HTML donde aparecerá el gráfico resultante.
visor_arbol = widgets.HTML(layout=widgets.Layout(width="100%"))


# Define la función que actualiza el gráfico cuando se cambia un control.
def actualizar_arbol(cambio=None):
    # Ignora notificaciones que no correspondan a cambios en el valor.
    if cambio is not None and cambio["name"] != "value":
        return
    # Si Ward coincide con Manhattan, vuelve a enlace completo, compatible con ambas distancias.
    if selector_distancia.value != "euclidean" and selector_enlace.value == "ward":
        selector_enlace.value = "complete"
        # El cambio del menú de enlace activará esta función de nuevo.
        return
    # Actualiza el contenido del visor con la combinación seleccionada.
    visor_arbol.value = explorar_arbol(selector_distancia.value, selector_enlace.value,
                                      selector_escala.value)


# Registra la función actualizadora para cada control de valor.
for control in (selector_distancia, selector_enlace, selector_escala):
    control.observe(actualizar_arbol, names="value")
# Calcula y muestra la combinación inicial de controles.
actualizar_arbol()
# Organiza título, menús y resultado en un solo panel vertical.
control_arbol = widgets.VBox([
    cabecera_widget("Explora distancias y enlaces"),
    fila_controles([selector_distancia, selector_enlace, selector_escala]),
    visor_arbol,
], layout=widgets.Layout(width="100%", grid_gap="12px"))
# Presenta el panel interactivo en el cuaderno.
display(control_arbol)

### 5.2.4. Calidad del árbol: correlación cofenética

Comparamos las distancias entre flores con las alturas a las que se unen, usando la fórmula de la exploración anterior.

`cophenet` obtiene las distancias cofenéticas y `np.corrcoef` calcula su correlación con las originales. Un valor próximo a 1 indica que el árbol refleja bien su relación lineal, **no que sus distancias sean iguales ni que un corte sea correcto**.

El umbral **0,75** es una orientación, no una prueba de validez. El enlace promedio puede dar una correlación alta, pero no está garantizado para cualquier conjunto de datos.

In [ ]:
# Calcula las distancias cofenéticas: altura a la que cada pareja se une en el árbol.
cop_dist = cophenet(hc)
# Correlaciona las distancias originales con las alturas cofenéticas para evaluar el árbol.

### 5.2.5. Explorar particiones: inercia y codo

La **inercia interna**, también llamada WSS, suma las distancias al cuadrado de las flores al centro de su grupo:

$$W_k=\sum_{g=1}^{k}\sum_{i\in g}\lVert x_i-\bar{x}_g\rVert^2.$$

$x_i$ contiene las medidas de una flor y $\bar{x}_g$ es la media de su grupo. Una WSS menor indica grupos más compactos. En Iris original se expresa en **cm²**.

La dispersión total se divide en interna y entre grupos:

$$T=W_k+B_k,\qquad \frac{B_k}{T}=1-\frac{W_k}{T}.$$

$B_k/T$ expresa cuánto se reduce la dispersión respecto a utilizar un solo centro. No es un porcentaje de aciertos. `resumen_particion` devuelve WSS en `inertia_`, $B_k$ en `betweenss` y $T$ en `totss`.

**Regla del codo:** busca una bajada grande de WSS seguida de mejoras menores. Aquí cortamos un árbol de enlace completo; no estamos ajustando k-means.

También conviene mirar los saltos del árbol, preferir una solución sencilla e interpretar los grupos. Un cociente de incrementos puede resumir los cambios, pero no lo usamos como regla automática: una cifra aislada no sustituye estas comprobaciones.

In [ ]:
# Calcula la WSS al cortar el mismo árbol completo desde uno hasta diez grupos.
inercia_hc = pd.Series({k: resumen_particion(iris2, cut_tree(hc, n_clusters=k).ravel()).inertia_
                        for k in range(1, 11)}, name="Inercia intra")
# Dibuja cómo cambia la inercia interna al aumentar el número de grupos.
ax = inercia_hc.plot(marker="o", color=COLOR_BASE)
# Etiqueta los ejes y pone un título que recuerda la regla del codo.
_ = ax.set(xlabel="Número de grupos", ylabel="Inercia intra (cm²)", title="Regla del codo")
# Añade una cuadrícula horizontal suave para facilitar la lectura.
_ = estilo_ejes(ax, cuadricula=True)
# Muestra el gráfico.
plt.show()
# Presenta los valores de WSS para k=1, 2, 3 y 4 con tres decimales.

**Codo del enlace completo:** las WSS para uno, dos, tres y cuatro grupos son **681,371; 235,153; 89,525 y 60,973 cm²**. Tres grupos es una opción razonable para explorar.

### 5.2.6. Cuatro índices para comparar particiones

Son **internos**: utilizan medidas y grupos, sin especies. Buscan compactación y separación. La conectividad pregunta si los vecinos quedan en el mismo grupo; estos índices no la miden directamente.

Evaluamos cortes de **Ward**, con las medidas originales:

- **Silhouette:** compara la distancia media de una flor a las demás de su grupo, $a(i)$, con la distancia media al otro grupo más cercano, $b(i)$:

  $$s(i)=\frac{b(i)-a(i)}{\max\{a(i),b(i)\}}.$$

  Va de −1 a 1. Alto indica mejor separación; cerca de 0, poca separación; negativo, mayor proximidad a otro grupo. `silhouette_score` promedia los valores de las flores. Si una flor está sola en su grupo, su valor se toma como 0.

- **Calinski–Harabasz (CH):**

  $$CH_k=\frac{B_k/(k-1)}{W_k/(n-k)}.$$

  Compara dispersión entre grupos e interna, considerando el número de flores $n$ y grupos $k$. **Mayor es preferible.** Usamos `calinski_harabasz_score`.

- **Davies–Bouldin (DB):**

  $$DB=\frac{1}{k}\sum_{g=1}^{k}\max_{h\ne g}\frac{S_g+S_h}{\lVert\bar{x}_g-\bar{x}_h\rVert}.$$

  $S_g$ es la distancia media de los miembros al centro de su grupo. Para cada grupo se toma la comparación menos favorable y después se promedian esas comparaciones. **Menor es preferible.** Usamos `davies_bouldin_score`.

- **Dunn:**

  $$Dunn=\frac{\min_{g\ne h}\;\min_{i\in g,\,j\in h}d(x_i,x_j)}{\max_g\;\max_{i,j\in g}d(x_i,x_j)}.$$

  Divide la menor distancia entre grupos por el mayor diámetro dentro de un grupo. **Mayor es preferible**, aunque una sola pareja puede influir mucho. `indice_dunn` recibe una matriz de distancias, preparada con `squareform(pdist(...))`, y las etiquetas.

Evaluamos entre **2 y $n-1$ grupos**, con dispersión interna no nula. WSS se interpreta con el codo. Los cuatro índices pueden favorecer distintos números de grupos: compara cada columna consigo misma. CH y DB aportan criterios complementarios a Silhouette y Dunn.

In [ ]:
# Define el índice Dunn y una función que compara particiones para distintos valores de k.
def indice_dunn(distancias, etiquetas):
    # Convierte las distancias a un array NumPy para poder seleccionar sus filas y columnas.
    D = np.asarray(distancias)
    # Obtiene los números de grupo presentes en la partición.
    grupos = np.unique(etiquetas)
    # Calcula el diámetro de cada grupo y conserva el mayor.
    diametro = max(D[np.ix_(etiquetas == g, etiquetas == g)].max() for g in grupos)
    # Busca la pareja más próxima de observaciones que pertenecen a grupos diferentes.
    separacion = min(
        # Selecciona las distancias entre las observaciones de los dos grupos.
        D[np.ix_(etiquetas == g, etiquetas == h)].min()
        # Recorre todas las combinaciones de dos grupos, sin repetirlas.
        for i, g in enumerate(grupos)
        for h in grupos[i + 1:]
    )
    # Devuelve separación entre grupos sobre diámetro interno; evita dividir por cero.
    return separacion / diametro if diametro > 0 else np.inf


# Define una función para calcular WSS y cuatro índices internos para varios valores de k.
def evaluar_indices(datos, metodo="ward", min_k=2, max_k=15, random_state=12345):
    # Convierte los datos a números decimales para los cálculos estadísticos.
    a = np.asarray(datos, dtype=float)
    # Calcula la matriz cuadrada de distancias entre observaciones.
    distancia = squareform(pdist(a))
    # Para k-means no hace falta un árbol; otros métodos construyen su jerarquía.
    arbol = None if metodo == "kmeans" else linkage(pdist(a), method=metodo)
    # Prepara una lista vacía donde guardar resultados de cada k.
    filas = []

    # Repite el cálculo para cada número de grupos, incluyendo min_k y max_k.
    for k in range(min_k, max_k + 1):
        # Ajusta k-means si ese es el método seleccionado.
        if metodo == "kmeans":
            # Guarda las etiquetas que devuelve el modelo con diez inicios.
            etiquetas = ajustar_kmeans(
                a, k, n_init=10, random_state=random_state
            ).labels_
        # Si no es k-means, corta el árbol que ya se construyó.
        else:
            # Renumera las etiquetas para que sean números consecutivos.
            etiquetas = renumerar(cut_tree(arbol, n_clusters=k).ravel())

        # Calcula resúmenes de dispersión para las etiquetas obtenidas.
        resumen = resumen_particion(a, etiquetas)
        # Añade WSS y los cuatro índices a la lista de resultados.
        filas.append({
            # Registra la suma de cuadrados dentro de los grupos.
            "WSS": resumen.inertia_,
            # Registra el índice promedio de Silhouette.
            "Silhouette": silhouette_score(a, etiquetas),
            # Registra el índice Calinski–Harabasz.
            "Calinski-Harabasz": calinski_harabasz_score(a, etiquetas),
            # Registra el índice Davies–Bouldin.
            "Davies-Bouldin": davies_bouldin_score(a, etiquetas),
            # Registra el índice Dunn usando la matriz de distancias.
            "Dunn": indice_dunn(distancia, etiquetas),
        })

    # Convierte los resultados en una tabla y usa k como nombre del índice de filas.
    return pd.DataFrame(
        filas, index=pd.Index(range(min_k, max_k + 1), name="k")
    )

In [ ]:
# Evalúa cortes de Ward de dos a quince grupos con las cuatro medidas originales.
NB = evaluar_indices(iris2, metodo="ward", min_k=2, max_k=15, random_state=SEED)
# Muestra la tabla de resultados redondeada a tres decimales.

#### ¿Qué número de grupos favorece cada criterio?

Para los cortes de Ward entre **2 y 15 grupos**:

| Criterio | Qué buscamos | Opción favorecida | Valor |
|:--|:--|:--:|--:|
| WSS | Codo | 3 como candidato visual | 79,297 |
| Silhouette | Máximo | 2 | 0,687 |
| Calinski–Harabasz | Máximo | 3 | 558,058 |
| Davies–Bouldin | Mínimo | 2 | 0,383 |
| Dunn | Máximo | 2 | 0,339 |

En Ward, WSS pasa de **154,947** con dos grupos a **79,297** con tres y **58,821** con cuatro. Conviene examinar **dos y tres grupos**. Compara cada columna consigo misma al variar $k$, no los valores de índices distintos.

### 5.2.7. Cortar el árbol completo en tres grupos

Volvemos a **`hc`: enlace completo y medidas originales**, no al Ward de la tabla.

`cut_tree` obtiene una etiqueta por flor; `renumerar` cambia sus nombres. Guardamos el corte de tres grupos en `ct`. `np.bincount` cuenta sus miembros: **50, 72 y 28**.

La proporción $B_3/T$ es **86,9 %**: reducción de dispersión respecto a un solo centro, no porcentaje de especies acertadas.

El deslizador muestra otros cortes sin modificar `ct`. **Sin controles:** utiliza `display(particion_jerarquica(3))`.

In [ ]:
# Obtiene y renumera las etiquetas del corte de tres grupos del árbol original.
ct = pd.Series(renumerar(cut_tree(hc, n_clusters=3).ravel()), index=iris.index, name="grupo")


# Define una función para mostrar un corte de k grupos sin cambiar el árbol guardado.
def particion_jerarquica(k):
    # Corta el árbol original en k grupos y asigna números sencillos a las etiquetas.
    grupos = renumerar(cut_tree(hc, n_clusters=k).ravel())
    # Crea una figura y un eje para dibujar el dendrograma.
    fig, ax = plt.subplots(figsize=(14, 5))
    # Dibuja el árbol y guarda el orden en que aparecen sus hojas.
    dibujado = dendrogram(hc, no_labels=True, link_color_func=lambda nodo: "#5B6770", ax=ax)
    # Convierte el orden de las hojas a un array NumPy para indexar etiquetas.
    orden = np.asarray(dibujado["leaves"])
    # Elige una altura intermedia entre las fusiones que rodean el corte solicitado.
    altura = (hc[-k, 2] + hc[-k+1, 2]) / 2
    # Dibuja un recuadro alrededor del tramo de hojas de cada grupo.
    for grupo in np.unique(grupos):
        # Encuentra las posiciones del grupo en el orden que muestra el dendrograma.
        posiciones = np.flatnonzero(grupos[orden] == grupo)
        # Elige el color asignado a este grupo.
        color = COLORES_GRUPO[int(grupo) % 10]
        # Añade un rectángulo sin relleno para resaltar sus hojas.
        ax.add_patch(Rectangle((10 * posiciones.min(), 0), 10 * len(posiciones), altura,
                               fill=False, edgecolor=color, linewidth=2))
    # Añade etiquetas y título que indican qué corte se está representando.
    ax.set(xlabel="Observaciones ordenadas por el árbol", ylabel="Distancia (cm)", title=f"Partición en {k} grupos")
    # Evita que el contenido se corte al mostrar la figura.
    fig.tight_layout()
    # Inserta la figura como imagen HTML dentro del cuaderno.
    display(widgets.HTML(value=imagen_control(fig, f"Dendrograma dividido en {k} grupos")))
    # Resume la partición con las medidas originales, no con los datos del dibujo.
    resumen = resumen_particion(iris2, grupos)
    # Imprime qué fracción de dispersión total separan los centros; no es exactitud.
    print(f"Reducción relativa de dispersión B/T (no acierto): {resumen.betweenss / resumen.totss:.1%}")
    # Devuelve cuántas flores hay en cada grupo.
    return pd.Series(np.bincount(grupos), index=np.arange(1, k+1), name="Tamaño del grupo")

In [ ]:
# Crea un control interactivo que vuelve a dibujar el árbol al cambiar k.
control_jerarquia = widgets.interactive(
    particion_jerarquica,
    k=widgets.IntSlider(value=3, min=2, max=8, description="Grupos", continuous_update=False),
)
# Presenta el deslizador y el gráfico en un panel con título.
display(panel_interactivo("Explora el corte del árbol", control_jerarquia))

### 5.2.8. Representar los tres grupos jerárquicos fijos

Recuperamos **`ct`**, el corte de enlace completo con medidas originales.

`pares_coloreados` dibuja pares de medidas en centímetros. Después, `representar_clusters` muestra dos proyecciones ACP: con medidas originales y estandarizadas para dibujar.

Busca dónde se separan o se solapan los colores. Las flores conservan su grupo en todas las vistas: cambiar el dibujo no vuelve a agruparlas.

In [ ]:
# Dibuja las parejas de medidas originales, coloreando cada flor según su etiqueta en ct.
pares_coloreados(iris2, ct.to_numpy())

#### ACP de las medidas originales

`PCA().fit(iris2)` resta las medias, pero no estandariza las columnas.

La primera componente conserva **92,46 %** de la variación y la segunda **5,31 %**: juntas, aproximadamente **97,77 %**. Esto describe la información conservada en el dibujo, no la calidad de los grupos.

Cada eje combina las cuatro medidas; no representa directamente una medida de sépalo o pétalo. Coloreamos con los grupos ya guardados en `ct`.

In [ ]:
# Ajusta el análisis de componentes principales a las medidas originales centradas.
pr = PCA().fit(iris2)
# Proyecta cada flor sobre los componentes calculados.
scores = pr.transform(iris2)
# Guarda las coordenadas de las dos primeras componentes para dibujar.
x, y = scores[:, 0], scores[:, 1]
# Crea un gráfico para mostrar cada flor en las dos componentes.
fig, ax = plt.subplots()
# Colorea cada punto con el grupo jerárquico ya fijado en ct.
_ = ax.scatter(x, y, c=ct, cmap=MAPA_GRUPOS, vmin=-0.5, vmax=9.5, s=24)
# Etiqueta los ejes y el título para indicar que la ACP usa escala original.
_ = ax.set(xlabel="CP1", ylabel="CP2", title="Grupos jerárquicos: ACP sin estandarizar")
# Ajusta los márgenes de la figura.
fig.tight_layout()
# Muestra el gráfico.
plt.show()
# Muestra qué porcentaje de variación explica cada uno de los dos ejes dibujados.
display(pd.Series(100 * pr.explained_variance_ratio_[:2], index=["CP1", "CP2"],
                  name="Varianza de la proyección (%)").round(2))
# Vuelve a dibujar las mismas etiquetas, estandarizando únicamente para proyectar.

### 5.2.9. Comparar los grupos con las especies: ARI

Ahora consultamos las especies, que no se utilizaron para agrupar. Es una comparación **externa**. `pd.crosstab` cuenta flores de cada especie y grupo. Tras emparejar los nombres:

| Especie | Grupo 1 | Grupo 2 | Grupo 3 |
|:--|--:|--:|--:|
| setosa | 50 | 0 | 0 |
| versicolor | 0 | 27 | 23 |
| virginica | 0 | 1 | 49 |

Coinciden **126 de 150 flores: 84 %**. Describe esta muestra, no el rendimiento con flores nuevas.

#### Rand, Jaccard y ARI: comparar parejas

Para las **mismas observaciones**, cuenta cada pareja una vez. Llamamos $a$ a las parejas juntas en ambas particiones, $b$ a las separadas en ambas, y $c,d$ a los dos tipos de desacuerdo.

$$Rand=\frac{a+b}{a+b+c+d},\qquad Jaccard=\frac{a}{a+c+d}.$$

Rand cuenta acuerdos al juntar **y** separar; Jaccard solo acuerdos al juntar. Van de 0 a 1. Si ninguna partición junta parejas, se toma Jaccard = 1. No calculamos estas dos medidas en las celdas siguientes; utilizamos ARI, que además corrige el azar.

#### Fórmula de ARI

El **índice de Rand ajustado** no necesita renombrar grupos:

$$ARI=\frac{S-E}{(A+B)/2-E},\qquad E=\frac{AB}{N}.$$

Aquí $S=a$; $A=a+c$ cuenta parejas juntas en la primera partición; $B=a+d$, juntas en la segunda; $N=a+b+c+d$ cuenta todas las parejas. **Las mayúsculas $A,B$ son totales de parejas, no los acuerdos $a,b$.** $E$ es el acuerdo esperado al azar conservando los tamaños.

Entre $m$ flores hay $\binom{m}{2}=m(m-1)/2$ parejas. Aquí **$S=3005$, $A=3675$, $B=4159$ y $N=11175$**. `adjusted_rand_score` calcula **ARI ≈ 0,642**.

- **1:** agrupaciones idénticas, aunque cambien sus nombres.
- **Cerca de 0:** acuerdo parecido al esperado por azar.
- **Negativo:** acuerdo menor que el esperado por azar.

No significa 64,2 % de aciertos. En el gráfico, **color = grupo** y **símbolo = especie**. Distintos símbolos del mismo color muestran mezcla de especies.

In [ ]:
# Cuenta la cantidad de flores de cada especie asignada a cada grupo de ct.
tabla_especies = pd.crosstab(iris["Species"], ct)

# Encuentra qué grupo conviene asociar con qué especie para maximizar coincidencias.
filas, columnas = linear_sum_assignment(-tabla_especies.to_numpy())
# Crea un diccionario que traduce los números de grupo a los nombres emparejados.
correspondencia = {
    tabla_especies.columns[columna]: int(fila)
    for fila, columna in zip(filas, columnas)
}
# Renombra las etiquetas para presentarlas; las flores no cambian de grupo.
cl = ct.map(correspondencia)

# Prepara una tabla cruzada con especies y grupos renombrados y la muestra.
t_pred = pd.crosstab(iris["Species"], cl + 1)
display(t_pred)
# Calcula la coincidencia descriptiva en esta muestra, no una prueba con datos nuevos.
print(f"Coincidencia descriptiva (no test): {np.trace(t_pred.to_numpy()) / t_pred.to_numpy().sum():.0%}")
# Calcula el acuerdo ajustado por azar entre grupos y especies, sin renombrar grupos.

In [ ]:
# Crea una cuadrícula de 16 paneles para comparar todas las parejas de medidas.
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
# Define un símbolo distinto para dibujar cada una de las tres especies.
marcadores = ["o", "s", "^"]
# Recorre las cuatro medidas para elegir qué variable va en cada fila.
for i, nombre_y in enumerate(iris2.columns):
    # Recorre las cuatro medidas para elegir qué variable va en cada columna.
    for j, nombre_x in enumerate(iris2.columns):
        # Selecciona el panel de la fila y columna actuales.
        ax = axes[i, j]
        # En la diagonal muestra la distribución de una medida con un histograma.
        if i == j:
            _ = ax.hist(iris2[nombre_x], bins=12, color=COLOR_BASE, alpha=0.75)
        # Fuera de la diagonal compara dos medidas mediante puntos.
        else:
            # Recorre las especies conocidas solo para cambiar el símbolo del dibujo.
            for especie, marcador in enumerate(marcadores):
                # Selecciona las flores que pertenecen a la especie actual.
                mascara = dataset.target == especie
                # Dibuja las flores con color de grupo y símbolo de especie.
                _ = ax.scatter(iris2.loc[mascara, nombre_x], iris2.loc[mascara, nombre_y],
                               c=cl.loc[mascara], cmap=MAPA_GRUPOS, vmin=-0.5, vmax=9.5,
                               marker=marcador, s=12)
        # Escribe el nombre de la medida horizontal solo en la fila inferior.
        if i == 3: _ = ax.set_xlabel(nombre_x)
        # Escribe el nombre de la medida vertical solo en la columna izquierda.
        if j == 0: _ = ax.set_ylabel(nombre_y)
# Ajusta espacios para que los paneles y las etiquetas quepan en la figura.
fig.tight_layout()
# Muestra la matriz; los grupos ya estaban calculados antes de este dibujo.

#### Micropráctica 2 (opcional) — cambiar enlace y comparar criterios

**Consulta:** enlaces y `linkage` en **5.2 y 5.2.2**; `cophenet` en **5.2.4**; `resumen_particion` en **5.2.5**; `cut_tree` en **5.2.7**; `adjusted_rand_score` en **5.2.9**.

Completa `enlace_practica` con el enlace promedio. Mantén las medidas originales, la distancia euclídea y tres grupos.

Compara cómo representa el árbol las distancias, la dispersión interna del corte y el acuerdo con `ct`.

Antes de ejecutar, explica por qué mejorar la correlación cofenética no obliga a reducir WSS ni a obtener ARI igual a 1. No utilices las especies para elegir el enlace.

In [ ]:
# Elige el enlace promedio de SciPy para el árbol de esta práctica.
enlace_practica = "average"
# Construye un árbol promedio usando las distancias euclídeas originales.
arbol_practica = linkage(d, method=enlace_practica)
# Corta el árbol en tres grupos y cambia sus etiquetas a números consecutivos.
grupos_practica = renumerar(cut_tree(arbol_practica, n_clusters=3).ravel())
# Resume la dispersión de esos grupos sobre las medidas originales.
resumen_practica = resumen_particion(iris2, grupos_practica)
# Reúne fidelidad, WSS, B/T y ARI en una serie para comparar criterios distintos.
resultados_practica = pd.Series({
    "Fidelidad cofenética frente a d": cophenet(arbol_practica, d)[0],
    "WSS (cm²)": resumen_practica.inertia_,
    "B/T (no acierto)": resumen_practica.betweenss / resumen_practica.totss,
    "ARI frente al corte completo ct": adjusted_rand_score(ct, grupos_practica),
})
# Muestra las cuatro medidas calculadas.
display(resultados_practica)
# Recuerda que representar distancias, compactación y coincidencia entre grupos son preguntas distintas.

## 5.3. Tendencia al agrupamiento: Hopkins

Hopkins compara los datos con puntos uniformes dentro de los mínimos y máximos de cada medida.

- $w_i$: distancia de una observación real a su vecino real más próximo, excluyéndose a sí misma.
- $u_i$: distancia de un punto uniforme a la observación real más próxima.

Usamos $H_0$ y mostramos también su complemento para reconocer las dos formas de expresar el índice:

$$H_0=\frac{\sum_i w_i}{\sum_i w_i+\sum_i u_i},\qquad 1-H_0=\frac{\sum_i u_i}{\sum_i w_i+\sum_i u_i}.$$

**Lectura orientativa de $H_0$:** cerca de **0**, los datos están más concentrados que la referencia uniforme; cerca de **0,5**, se parecen a ella; cerca de **1**, pueden estar más regularmente separados. Si se usa $1-H_0$, la concentración se lee **hacia 1**. Ambas fracciones contienen la misma información, pero se interpretan en sentidos opuestos.

`hopkins` recibe los datos, el tamaño de muestra `n` y la semilla. `n` indica filas muestreadas, no grupos. Usamos distancias euclídeas sin elevarlas al número de variables, tal como aparecen en la fórmula anterior. El resultado depende de escala y muestreo; **no determina $k$ ni prueba que existan grupos reales**.

In [ ]:
# Define una función que compara la concentración de datos reales con puntos uniformes.
def hopkins(datos, n=None, random_state=12345):
    # Convierte los datos a números decimales para medir distancias.
    a = np.asarray(datos, dtype=float)
    # Elige cuántas filas reales muestrear; por defecto usa todas menos una.
    m = len(a) - 1 if n is None else int(n)
    # Comprueba que la muestra tiene al menos una fila y menos filas que los datos.
    if not 1 <= m < len(a):
        # Explica el rango permitido cuando el tamaño de muestra no es válido.
        raise ValueError("El tamaño de muestra debe estar entre 1 y n-1.")
    # Crea un generador aleatorio con semilla para repetir la misma muestra.
    rng = np.random.default_rng(random_state)
    # Elige índices de filas reales sin repetirlos.
    indices = rng.choice(len(a), m, replace=False)
    # Calcula distancias entre las filas elegidas y todas las filas reales.
    reales = cdist(a[indices], a)
    # Excluye cada fila de sí misma, para que no tenga distancia cero consigo.
    reales[np.arange(m), indices] = np.inf
    # Guarda para cada fila muestreada la distancia a su vecino real más próximo.
    w = reales.min(axis=1)
    # Genera puntos de referencia uniformes dentro del rango observado de cada variable.
    uniformes = rng.uniform(a.min(axis=0), a.max(axis=0), size=(m, a.shape[1]))
    # Calcula para cada punto uniforme la distancia a la observación real más cercana.
    u = cdist(uniformes, a).min(axis=1)
    # Devuelve H0: valores pequeños sugieren concentración frente a la caja uniforme.
    return w.sum() / (w.sum() + u.sum())

In [ ]:
# Calcula Hopkins con 149 filas reales, las cuatro medidas originales y la semilla fija.
hop = hopkins(iris2, n=len(iris2) - 1, random_state=SEED)
# Muestra el valor calculado de H0.
hop
# Informa tanto H0 como su complemento, que se interpreta en la dirección opuesta.
print(f"Convención H0: {hop:.3f}; convención complementaria: {1-hop:.3f}")

**Resultado:** Iris da **$H_0=0,168$**, o **$1-H_0=0,832$**. Las flores tienen vecinos más próximos que los puntos de la referencia uniforme.

Esa referencia no conserva las relaciones entre medidas; el resultado no demuestra que existan tres grupos.

#### Micropráctica 3 (opcional) — muestra y escala

**Consulta:** `escalar` y estandarización en **5.0.1 y 5.1.2**; `hopkins` e interpretación en **5.3**.

Completa el tamaño muestral con **30**. Calcula Hopkins para Iris original y estandarizado, manteniendo la semilla, y expresa ambos resultados como $H_0$ y $1-H_0$.

Compara con el ejemplo de 149 filas. Explica qué cambió, sin utilizar Hopkins para escoger el número de grupos.

In [ ]:
# Usa una muestra de 30 flores para medir la tendencia al agrupamiento en escala original.
# random_state mantiene iguales las filas y los puntos aleatorios entre ejecuciones.
hop_original = hopkins(iris2, n=30, random_state=SEED)
# Repite la medición con las cuatro variables estandarizadas.
hop_estandarizado = hopkins(iris2_escalado, n=30, random_state=SEED)
# Junta ambos resultados para mostrarlos con nombres claros.
hop_practica = pd.Series({"Original": hop_original, "Estandarizado": hop_estandarizado})
# Crea una tabla con las dos convenciones: H0 y su complemento.
comparacion_hop = pd.DataFrame({
    "H0 (concentración hacia 0)": hop_practica,
    "1-H0 (concentración hacia 1)": 1 - hop_practica,
})
# Muestra ambos índices redondeados a tres decimales.

## 5.4. Ejercicio 5-1 — Vinos

**Objetivo:** aplicar a los vinos los pasos practicados con Iris. La carga está preparada; escribe las soluciones en las celdas de trabajo.

### 5.4.1. Revisar la escala

Ejecuta la carga de las 13 medidas en `vinos`; `tipos_vino` queda reservado para el final. Muestra primeras filas, medias y desviaciones típicas. Estandariza las medidas y comprueba su media y desviación típica.

**Consulta:** `head()` y `describe()` en **5.1 y 5.1.1**; `escalar` en **5.0.1 y 5.1.2**. Guarda medidas originales y estandarizadas por separado, sin incluir los tipos conocidos.

**Pregunta:** cita dos variables con escalas diferentes y explica cómo pueden afectar a las distancias.

In [ ]:
# Carga las 13 medidas de cada vino desde el archivo de datos.
vinos = pd.read_csv(DATA_PATH / "wine2.txt")
# Carga la tabla que también contiene los tipos conocidos, reservados para el final.
vinos_con_tipo = pd.read_csv(DATA_PATH / "wine.data.txt")
# Separa la columna de tipos para no usarla durante el clustering.
tipos_vino = vinos_con_tipo["Type"].copy()
# Comprueba que hay 178 vinos, 13 variables y ningún dato ausente.
assert vinos.shape == (178, 13) and not vinos.isna().any().any()
# Verifica que las medidas coinciden entre los dos archivos.
assert np.allclose(vinos.to_numpy(), vinos_con_tipo.iloc[:, 1:].to_numpy())

# Espacio para tu solución de 5.4.1: inspecciona medias, desviaciones y escalas.
# Consulta head(), describe() y escalar en las secciones indicadas.
# Guarda las medidas estandarizadas aparte y comprueba su media y desviación típica.
# Explica por qué variables con escalas distintas pueden influir de forma diferente en distancias.
# No uses tipos_vino para estandarizar ni para ajustar grupos.

### 5.4.2. Construir árboles y obtener tres grupos

Con las medidas **estandarizadas**, compara:
- Distancia euclídea y enlace completo.
- Distancia euclídea y Ward.
- Distancia Manhattan y enlace completo.

Dibuja cada dendrograma, córtalo en **tres grupos** y cuenta sus miembros.

**Consulta:** `pdist`, `linkage` y `dendrogram` en **5.2.1–5.2.3**; `cut_tree` y `np.bincount` en **5.2.7**. Los nombres de distancias son `euclidean` y `cityblock`; los enlaces, `complete` y `ward`. Ward requiere euclídea.

**Pregunta:** ¿cambian los tamaños al cambiar enlace o distancia? Los números de grupo son solo etiquetas.

In [ ]:
# Apartado 5.4.2: crea un diccionario para guardar por separado árboles y etiquetas.
# Usa vinos estandarizados y distancia euclídea con enlace completo.
# Repite el árbol con distancia euclídea y enlace Ward.
# Repite con distancia Manhattan (cityblock) y enlace completo.
# Dibuja cada árbol, córtalo en tres grupos y cuenta sus miembros.
# Guarda cada resultado de forma separada para comparar tamaños y pertenencias.
# No uses los tipos conocidos para decidir qué agrupación conservar.

### 5.4.3. Comparar e interpretar

1. Usa el corte **euclídea/completo** como referencia y calcula su ARI con cada corte alternativo.
2. Calcula WSS y $B/T$ de la referencia con las medidas estandarizadas.
3. Representa sus grupos en dos componentes principales.
4. **Solo al final**, cruza `tipos_vino` con los grupos.

**Consulta:** `adjusted_rand_score` y `pd.crosstab` en **5.2.9**; `resumen_particion` en **5.0.1, 5.2.5 y 5.2.7**; `representar_clusters` en **5.0.2 y 5.2.8**.

**Responde:** ¿qué corte se parece más a la referencia? ¿Qué significa $B/T$? ¿Los grupos reúnen un tipo de vino o mezclan tipos?

Reserva los tipos conocidos para esta comparación final, no para escoger distancia, enlace o número de grupos.

In [ ]:
# Apartado 5.4.3: elige el corte euclídeo/completo como referencia.
# Calcula ARI entre esa referencia y cada corte alternativo.
# Calcula WSS y B/T con las medidas estandarizadas de referencia.
# Dibuja en dos componentes principales las etiquetas ya calculadas.
# Consulta tipos_vino solamente en la tabla cruzada final.
# Interpreta el ARI como acuerdo entre particiones, no como porcentaje de acierto.
# Explica B/T como reducción relativa de dispersión y revisa si los tipos se mezclan.

#### Ampliación opcional — pasar de tres a cuatro grupos

**Consulta:** `cut_tree` y `np.bincount` en **5.2.7**; `resumen_particion` y $B/T$ en **5.2.5**; `adjusted_rand_score` y comparación de cortes en **5.4.3**.

Corta el mismo árbol euclídea/completo en **cuatro grupos**. Mantén los datos estandarizados. Compara tamaños, $B/T$ y ARI con el corte de tres grupos.

**Pregunta:** si aumenta $B/T$, ¿basta para afirmar que cuatro grupos es mejor? Justifica sin consultar los tipos conocidos.

## 5.5. K-means: agrupar alrededor de centros

Antes elegimos **$k$**, el número de grupos. Cada grupo tiene un **centroide**: la media de las medidas de sus miembros. En Iris es una «flor promedio», no necesariamente una flor real.

**Lloyd, paso a paso:**
1. Elegir $k$ centros iniciales.
2. Asignar cada observación al centro más próximo.
3. Recalcular los centros como medias de sus grupos.
4. Repetir hasta estabilizar los centros o alcanzar el límite de iteraciones.

El objetivo es reducir **WSS** (**5.2.5**). Que el algoritmo se estabilice **no garantiza la mejor solución posible**.

**Hartigan–Wong** busca ese mismo objetivo trasladando observaciones entre grupos cuando reduce WSS y actualizando las medias. El primer ejemplo usa la versión didáctica del proyecto; sus transferencias no siguen exactamente la secuencia de asignación y actualización de Lloyd.

Los inicios son **aleatorios** en nuestra función, tanto con Lloyd como con Hartigan–Wong. Otra estrategia, **k-means++**, elige centros iniciales favoreciendo puntos alejados de los ya seleccionados. La misma semilla, por ejemplo **42**, no garantiza los mismos centros con estrategias diferentes: comparar ajustes con distintos algoritmos e inicios mezcla ambos efectos. Para aislar el algoritmo habría que proporcionar los mismos centros mediante `init`.

Varios inicios ayudan a encontrar una WSS menor. La escala y los valores extremos pueden cambiar los grupos; formas como anillos no se representan bien con estos centros. K-means no construye un árbol ni necesita todas las distancias por parejas.

### 5.5.1. Primer ejemplo: tres grupos de flores

`ajustar_kmeans` agrupa Iris en **tres grupos**, con medidas originales y un único inicio. Observamos etiquetas (`labels_`), centros (`cluster_centers_`), tamaños y WSS (`inertia_`). Los números de grupo no representan especies.

In [ ]:
# Ajusta k-means de Hartigan–Wong a las cuatro medidas originales para tres grupos.
km0 = ajustar_kmeans(iris2, k=3, n_init=1, random_state=SEED)
# Convierte las etiquetas a una Serie e identifica cada flor con su índice.
grupos_km0 = pd.Series(km0.labels_ + 1, index=iris2.index, name="Grupo")
# Muestra la etiqueta arbitraria asignada a cada flor.

In [ ]:
# Muestra las medias de las medidas de las flores en cada grupo; las unidades son cm.
pd.DataFrame(km0.cluster_centers_, columns=iris2.columns, index=np.arange(1, 4))
# Estos centroides resumen grupos y no tienen por qué coincidir con una flor real.

In [ ]:
# Resume en detalle la partición de tres grupos usando los mismos datos del ajuste.
resumen_km0 = resumen_particion(iris2, km0.labels_)
# Muestra T: dispersión cuadrática respecto a la media global, en cm².
resumen_km0.totss
# Muestra WSS por grupo: dispersión de cada grupo respecto a su centro.
resumen_km0.withinss
# Muestra WSS total, que k-means intenta reducir.
resumen_km0.inertia_
# Muestra B: dispersión entre centros, calculada como T menos WSS.
resumen_km0.betweenss
# Muestra cuántas flores quedaron en cada grupo.
resumen_km0.size
# Muestra cuántas iteraciones realizó el ajuste.
km0.n_iter_
# Comprueba que la suma de las WSS parciales es la WSS total.
assert np.isclose(resumen_km0.withinss.sum(), resumen_km0.inertia_)
# Comprueba la identidad de descomposición T = W + B.
assert np.isclose(resumen_km0.totss, resumen_km0.inertia_ + resumen_km0.betweenss)

#### Leer $B/T$: reducción de dispersión

$$\frac{B}{T}=1-\frac{W}{T}.$$

Aquí es aproximadamente **88,4 %**: utilizar tres centros reduce esa proporción de dispersión respecto a un solo centro.

No es un porcentaje de especies acertadas. Añadir grupos puede aumentar $B/T$, por lo que no elegimos únicamente su valor máximo.

In [ ]:
# Calcula la fracción de dispersión entre grupos respecto a la dispersión total.
# Es una medida de compactación/separación, no un porcentaje de especies acertadas.
resumen_km0.betweenss / resumen_km0.totss

### 5.5.2. Número de grupos: dos formas de leer el codo

Ajustamos de **uno a diez grupos**, con diez inicios por valor de $k$.

Dibujamos WSS y $B_k/T$. Muestran el mismo cambio:

$$B_k/T=1-W_k/T.$$

Busca una mejora grande seguida de mejoras menores. Las primeras WSS son **681,371; 152,348; 78,851 y 57,228 cm²**.

Con tres grupos, k-means obtiene **78,851**, frente a **89,525** del corte de enlace completo. K-means busca reducir precisamente esta dispersión; eso no demuestra mayor acuerdo con las especies.

Compara WSS solo entre ajustes sobre los mismos datos y escala.

#### Codo como proporción de dispersión reducida

In [ ]:
# Ajusta modelos de k-means con entre 1 y 10 grupos y conserva cada resultado.
modelos_k = {k: ajustar_kmeans(iris2, k=k, n_init=10, random_state=SEED) for k in range(1, 11)}
# Calcula B/T para k desde 2; usa T del resumen de los mismos datos Iris.
VE = pd.Series({k: resumen_particion(iris2, modelos_k[k].labels_).betweenss / resumen_km0.totss
                for k in range(2, 11)}, name="Reducción relativa de dispersión B/T")
# Dibuja cómo aumenta la reducción relativa de dispersión al añadir grupos.
ax = VE.plot(marker="o", color=COLOR_BASE)
# Explica los ejes y titula el gráfico como una exploración del codo.
_ = ax.set(xlabel="Número de grupos", ylabel="Reducción relativa B/T (no acierto)", title="Regla del codo")
# Añade una cuadrícula horizontal para leer mejor las alturas.
_ = estilo_ejes(ax, cuadricula=True)
# Muestra la curva en el cuaderno.

#### Regla del codo: inercia dentro de los grupos

In [ ]:
# Recoge la inercia WSS de cada ajuste de k-means ya calculado.
inercia_k = pd.Series({k: modelo.inertia_ for k, modelo in modelos_k.items()})
# Dibuja la WSS según el número de grupos.
ax = inercia_k.plot(marker="o", color=COLOR_BASE)
# Etiqueta ejes y título para facilitar la interpretación del codo.
_ = ax.set(xlabel="Número de grupos", ylabel="Inercia interna WSS (cm²)", title="Regla del codo")
# Añade cuadrícula horizontal a la gráfica.
_ = estilo_ejes(ax, cuadricula=True)
# Muestra el gráfico.

#### Comparar los cuatro índices internos en k-means

La tabla aplica los criterios de **5.2.6**:
- **Mayor:** Silhouette, Calinski–Harabasz y Dunn.
- **Menor:** Davies–Bouldin.
- **WSS:** buscar el codo.

Compara cada columna al cambiar $k$. Los índices pueden discrepar; Dunn es especialmente sensible a unas pocas distancias.

In [ ]:
# Calcula cinco indicadores internos para k-means desde dos hasta quince grupos.
ncluster = evaluar_indices(iris2, metodo="kmeans", min_k=2, max_k=15, random_state=SEED)
# Muestra la tabla de resultados redondeada a tres decimales.

#### Conclusión: ¿cuántos grupos elegir?

En los ajustes de k-means evaluados:

| Criterio | Número de grupos favorecido | Valor |
|:--|:--:|--:|
| Codo de WSS | 3 como candidato visual | 78,851 |
| Silhouette | 2 | 0,681 |
| Calinski–Harabasz | 3 | 561,628 |
| Davies–Bouldin | 2 | 0,404 |
| Dunn | 12 | 0,154 |

Compararemos **dos y tres grupos**: dos ofrece mejor Silhouette y Davies–Bouldin; tres obtiene mejor CH y aparece como candidato en el codo. WSS baja de **152,348** a **78,851** y después a **57,228** con cuatro grupos.

Dunn favorece doce, pero ese resultado necesita más examen: unas pocas distancias influyen mucho en el índice.

Los gráficos siguientes ayudan a comparar separación, tamaños y detalle. Ningún indicador decide por sí solo el número de grupos.

#### Representación: tres grupos

Agrupamos las 150 flores con sus cuatro medidas originales. **Cada punto es una flor y cada color un grupo**, no una especie.

- En los **gráficos de pares**, compara dos medidas en centímetros.
- En el **ACP**, dos ejes resumen las cuatro medidas sin cambiar los grupos.
- Los **contornos** muestran la zona ocupada por cada grupo.

Una nube aparece apartada y las otras dos quedan más próximas. El ajuste utiliza cuatro medidas, aunque cada dibujo muestre solo dos.

**Pregunta de observación:** ¿el tercer grupo muestra una separación clara o divide una nube de flores parecidas? Consulta los índices de **5.2.6** y las vistas con `pares_coloreados` y `representar_clusters` de **5.0.2**.

In [ ]:
# Ajusta k-means con tres grupos, diez inicios y semilla reproducible.
km3 = ajustar_kmeans(iris2, k=3, n_init=10, random_state=SEED)
# Dibuja las parejas de medidas y colorea cada punto con el grupo ajustado.

In [ ]:
# Proyecta y dibuja en dos componentes principales las etiquetas calculadas en km3.
# Los contornos ayudan a visualizar la extensión aproximada de cada grupo.
representar_clusters(iris2, km3.labels_, titulo="K-means: tres grupos y envolventes convexas")

#### Representación: dos grupos

Repetimos k-means con las mismas medidas, pero pedimos **dos grupos**. Es un ajuste nuevo, no la unión automática de dos grupos anteriores.

**Punto = flor; color = grupo.** Los colores no tienen por qué representar los mismos conjuntos que en el ajuste de tres grupos.

- **Dos grupos:** resumen más sencillo y mayor Silhouette: **0,681 frente a 0,553**.
- **Tres grupos:** más detalle y menor WSS: **78,851 frente a 152,348 cm²**.

**Pregunta de observación:** ¿ese detalle adicional muestra una separación clara? Consulta los criterios de **5.2.6**, el codo de **5.5.2** y `representar_clusters` en **5.0.2**.

In [ ]:
# Ajusta otro modelo k-means, esta vez solicitando dos grupos.
km2 = ajustar_kmeans(iris2, k=2, n_init=10, random_state=SEED)
# Dibuja las parejas de medidas coloreadas según las etiquetas de este nuevo ajuste.

In [ ]:
# Crea una figura para mostrar las dos primeras componentes principales.
fig, ax = plt.subplots()
# Colorea cada flor con la etiqueta del ajuste k-means de dos grupos.
_ = ax.scatter(x, y, c=km2.labels_, cmap=MAPA_GRUPOS, vmin=-0.5, vmax=9.5, s=24)
# Etiqueta los ejes y el título del gráfico.
_ = ax.set(xlabel="CP1", ylabel="CP2", title="K-means: dos grupos")
# Ajusta los márgenes de la figura.
fig.tight_layout()
# Muestra la proyección.

In [ ]:
# Dibuja las etiquetas ya calculadas para km2 en dos componentes principales.
# La proyección facilita ver grupos, pero no vuelve a ajustar sus miembros.
representar_clusters(iris2, km2.labels_, titulo="K-means: dos grupos y envolventes convexas")

#### Explorar $k$, los inicios y el algoritmo

Cambia una opción cada vez:
- **Grupos:** cuántos grupos ajustar.
- **Inicios:** cuántos puntos de partida probar (`n_init`).
- **Algoritmo:** Lloyd o Hartigan–Wong (`algoritmo`).

Observa el dibujo, WSS y Silhouette. Para un mismo $k$, compara si una WSS menor también va acompañada de mayor Silhouette.

**Consulta:** `ajustar_kmeans` en **5.0.1 y 5.5.1**; `silhouette_score` en **5.2.6**. El control no sustituye los ajustes fijos anteriores.

In [ ]:
# Define una función que ajusta y resume k-means con las opciones elegidas en controles.
def explorar_kmeans(k, inicios, algoritmo):
    # Ajusta k-means con el número de grupos, inicios y algoritmo seleccionados.
    modelo = ajustar_kmeans(iris2, k, n_init=inicios, algoritmo=algoritmo, random_state=SEED)
    # Crea una representación HTML de las etiquetas para incluirla en el widget.
    grafico = representar_clusters(iris2, modelo.labels_, titulo=f"K-means: {k} grupos", mostrar=False)
    # Muestra el gráfico dentro de la salida interactiva.
    display(widgets.HTML(value=grafico))
    # Resume la dispersión de la partición calculada.
    resumen = resumen_particion(iris2, modelo.labels_)
    # Muestra WSS, B/T y Silhouette para comparar este ajuste.
    display(pd.Series({"WSS (cm²)": modelo.inertia_, "Reducción relativa B/T (no acierto)": resumen.betweenss / resumen.totss,
                       "Silhouette": silhouette_score(iris2, modelo.labels_)}))

In [ ]:
# Crea controles para explorar número de grupos, inicios y algoritmo k-means.
control_kmeans = widgets.interactive(
    explorar_kmeans,
    k=widgets.IntSlider(value=3, min=2, max=8, description="Grupos", continuous_update=False),
    inicios=widgets.SelectionSlider(options=[1, 5, 10], value=10, description="Inicios", continuous_update=False),
    algoritmo=widgets.Dropdown(options=[("Hartigan–Wong", "hartigan-wong"), ("Lloyd", "lloyd")], description="Algoritmo"),
)
# Presenta los controles y sus resultados en un panel titulado.
display(panel_interactivo("Compara los ajustes de k-means", control_kmeans))

### 5.5.3. Concordancia de k-means con las especies de Iris

Comparamos los tres grupos de `km3` con `Species` mediante `adjusted_rand_score` (**5.2.9**). Las especies son una referencia externa, no se utilizaron para agrupar. ARI mide acuerdo, no un porcentaje de aciertos.

### 5.5.4. Combinar métodos sin confundir ajuste y dibujo

Podemos combinar k-means y jerarquía en dos órdenes:
- **K-means → jerarquía:** formar muchos grupos pequeños y después agrupar sus centros. Resume un conjunto grande, pero los centros representan grupos de tamaños posiblemente distintos.
- **Jerarquía → k-means:** cortar un árbol y utilizar las medias de sus grupos como centros iniciales. K-means puede cambiar las pertenencias para reducir WSS. Practicarás esta opción con Ward en el mayorista; `init` recibe esos centros (**5.0.1**).

También se pueden resumir las variables antes de agrupar: **ACP** para medidas numéricas y **ACM** para categorías. Aquí el **ACP solo dibuja grupos ya calculados** (**5.0.2 y 5.2.8**), no decide sus miembros. Guardar solo dos componentes para ajustar podría perder información; una proyección clara no demuestra por sí sola una buena agrupación.

In [ ]:
# Calcula, solo como comparación externa posterior, el ARI entre grupos y especies.
# Las etiquetas de especie no se usaron al ajustar el modelo km3.
adjusted_rand_score(km3.labels_, iris["Species"])

## 5.6. Grupos no convexos: disco, corona y kernel

Creamos **500 puntos en un disco** de radio entre 0 y 1 y **500 en una corona** de radios entre 2 y 3.

Elegimos radio y ángulo al azar. Los radios son uniformes, pero los puntos no se reparten uniformemente por superficie: se concentran más hacia las zonas interiores.

Este ejemplo muestra una limitación de k-means: separar una nube central de otra que la rodea.

Agrupamos con coordenadas estandarizadas y dibujamos las originales, usando la misma escala en ambos ejes para conservar la forma circular.

In [ ]:
# Elige el número de puntos de cada forma circular que se va a simular.
n = 500
# Crea un generador aleatorio reproducible con la semilla general.
rng = np.random.default_rng(SEED)
# Genera radios uniformes entre cero y uno para el disco interior.
r1 = rng.uniform(0, 1, n)
# Genera un ángulo aleatorio para cada punto del disco interior.
alfa1 = rng.uniform(0, 2 * np.pi, n)
# Convierte radio y ángulo en coordenadas cartesianas x e y.
x1, y1 = r1 * np.cos(alfa1), r1 * np.sin(alfa1)
# Genera radios uniformes entre dos y tres para la corona exterior.
r2 = rng.uniform(2, 3, n)
# Genera un ángulo aleatorio para cada punto de la corona.
alfa2 = rng.uniform(0, 2 * np.pi, n)
# Convierte las coordenadas polares de la corona a coordenadas cartesianas.
x2, y2 = r2 * np.cos(alfa2), r2 * np.sin(alfa2)
# Combina las coordenadas de ambos conjuntos en una tabla para dibujar.
df_anillos = pd.DataFrame({"x": np.r_[x1, x2], "y": np.r_[y1, y2]})
# Estandariza por columna para usar una escala común en los ajustes.
df_anillos_escalado = escalar(df_anillos)

In [ ]:
# Prepara una figura cuadrada para que el dibujo conserve la forma circular.
fig, ax = plt.subplots(figsize=(6, 6))
# Dibuja todos los puntos simulados en un color, sin mostrar sus etiquetas de origen.
_ = ax.scatter(df_anillos["x"], df_anillos["y"], s=10, color=COLOR_BASE)
# Etiqueta los ejes y pone un título descriptivo.
_ = ax.set(xlabel="x", ylabel="y", title="Disco y corona circular")
# Usa la misma escala horizontal y vertical para no deformar círculos.
_ = ax.set_aspect("equal")
# Ajusta los márgenes de la figura.
fig.tight_layout()
# Muestra la nube de puntos.
plt.show()

In [ ]:
# Ajusta k-means con dos grupos sobre las coordenadas ya estandarizadas.
km_anillos = ajustar_kmeans(df_anillos_escalado, k=2, n_init=1, random_state=SEED)
# Prepara un gráfico de las coordenadas originales para interpretar la forma.
fig, ax = plt.subplots(figsize=(6, 6))
# Colorea cada punto con el grupo que le asignó k-means.
_ = ax.scatter(df_anillos["x"], df_anillos["y"], c=km_anillos.labels_,
               cmap=MAPA_GRUPOS, vmin=-0.5, vmax=9.5, s=10)
# Etiqueta el gráfico con su contenido y sus ejes.
_ = ax.set(title="K-means en grupos no convexos", xlabel="x", ylabel="y")
# Conserva la misma escala para los ejes y evitar distorsionar círculos.
_ = ax.set_aspect("equal")
# Ajusta el espacio alrededor del gráfico.
fig.tight_layout()
# Muestra la asignación de k-means.
plt.show()

### K-means con kernel gaussiano

Con dos centros, k-means euclídeo separa las regiones mediante una recta. Eso no permite distinguir un disco de una corona que lo rodea.

El **kernel gaussiano** asigna una similitud a cada pareja:

$$K(x_i,x_j)=\exp(-\gamma\lVert x_i-x_j\rVert^2),\qquad \gamma=\frac{1}{2\sigma^2}.$$

Aquí las distancias usan coordenadas estandarizadas. $\sigma$ indica el alcance de la similitud:
- **$\gamma$ menor:** alcanza distancias mayores.
- **$\gamma$ mayor:** se concentra en vecinos más próximos.

El código usa la matriz de similitudes para obtener un punto de partida (**inicialización espectral**) y después reajusta los grupos con k-means de kernel. No es simplemente k-means aplicado a las dos coordenadas. Los cálculos largos de esa inicialización no son materia del ejercicio.

El coste se mide en la representación del kernel, **no como WSS de las coordenadas originales**. Este ejemplo calcula una matriz de todas las parejas y resulta más costoso que k-means ordinario.

Esta **ampliación del cuaderno** ilustra una alternativa para formas no convexas, como el disco y la corona. DBSCAN, descrito en la lectura opcional del cierre, aborda estas formas mediante densidad en lugar de similitudes de kernel.

El ejemplo fijo usa **$\gamma=2$**. Para explorar, consulta `escalar` (**5.0.1**) y la llamada fija a `kernel_kmeans` de esta sección. Observa si distingue el disco de la corona; aumentar $\gamma$ no garantiza mejorar.

In [ ]:
# Define k-means con kernel: agrupa mediante similitudes en vez de distancias euclídeas directas.
def kernel_kmeans(datos, k=2, gamma=2.0, random_state=12345, max_iter=100):
    # Convierte los datos a números decimales para calcular distancias.
    a = np.asarray(datos, dtype=float)
    # Calcula la matriz de similitud gaussiana entre cada pareja de observaciones.
    K = np.exp(-gamma * squareform(pdist(a, metric="sqeuclidean")))
    # Normaliza las similitudes para preparar una representación espectral inicial.
    pesos = np.sqrt(K.sum(axis=1))
    # Divide cada similitud por el producto de los pesos de sus dos observaciones.
    normalizada = K / np.outer(pesos, pesos)
    # Limita hilos y obtiene los autovectores de mayor valor para inicializar los grupos.
    with threadpool_limits(limits=1):
        # Calcula los k autovectores principales de la matriz normalizada.
        _, vectores = eigh(normalizada, subset_by_index=[len(a) - k, len(a) - 1])
        # Normaliza cada fila para que cada punto tenga longitud uno.
        vectores = vectores / np.linalg.norm(vectores, axis=1, keepdims=True)
        # Usa k-means en la representación espectral para obtener etiquetas iniciales.
        etiquetas = KMeans(k, n_init=10, random_state=random_state).fit_predict(vectores)
    # Crea una lista para guardar el coste de cada iteración.
    historial = []
    # Repite las reasignaciones hasta estabilizarse o alcanzar el máximo de iteraciones.
    for iteracion in range(max_iter):
        # Reserva una columna de distancias en el espacio del kernel por cada grupo.
        distancias = np.empty((len(a), k))
        # Calcula la distancia de cada observación al centro implícito de cada grupo.
        for grupo in range(k):
            # Encuentra los índices de observaciones que pertenecen al grupo actual.
            miembros = np.flatnonzero(etiquetas == grupo)
            # Impide seguir si la inicialización produjo un grupo vacío.
            if not len(miembros):
                # Sugiere cambiar gamma para tratar de obtener una partición válida.
                raise ValueError("El kernel ha producido un grupo vacío; cambia gamma.")
            # Calcula distancias cuadráticas en el espacio implícito del kernel.
            distancias[:, grupo] = np.diag(K) - 2 * K[:, miembros].mean(axis=1) + K[np.ix_(miembros, miembros)].mean()
        # Asigna cada fila al grupo cuya distancia en kernel sea menor.
        nuevas = distancias.argmin(axis=1)
        # Registra el coste de las asignaciones actuales.
        historial.append(float(distancias[np.arange(len(a)), etiquetas].sum()))
        # Termina cuando ninguna etiqueta cambia.
        if np.array_equal(nuevas, etiquetas):
            break
        # Termina si la nueva asignación deja algún grupo vacío.
        if len(np.unique(nuevas)) < k:
            break
        # Guarda las nuevas etiquetas para la siguiente iteración.
        etiquetas = nuevas
    # Devuelve etiquetas, historial de costes y cantidad de iteraciones.
    return SimpleNamespace(labels_=etiquetas, history_=np.array(historial), n_iter_=iteracion + 1)

In [ ]:
# Calcula un resultado fijo de referencia con gamma 2 y lo conserva en kk.
kk = kernel_kmeans(df_anillos_escalado, k=2, gamma=2.0, random_state=SEED)

# Define una función que recalcula y dibuja el resultado para cada gamma elegido.
def explorar_kernel(gamma):
    # Ajusta el kernel con el valor seleccionado, manteniendo escala, k y semilla.
    resultado = kernel_kmeans(df_anillos_escalado, k=2, gamma=gamma, random_state=SEED)
    # Crea una figura cuadrada para mostrar la forma original del conjunto.
    fig, ax = plt.subplots(figsize=(6, 6))
    # Colorea las coordenadas originales según las nuevas etiquetas del modelo.
    _ = ax.scatter(df_anillos["x"], df_anillos["y"], c=resultado.labels_,
                   cmap=MAPA_GRUPOS, vmin=-0.5, vmax=9.5, s=10)
    # Construye un título que incluye el valor numérico de gamma.
    titulo = f"K-means con kernel gaussiano: gamma = {gamma:.2f}"
    # Etiqueta los ejes y pone el título en la figura.
    _ = ax.set(xlabel="x", ylabel="y", title=titulo)
    # Conserva la proporción de las coordenadas para no deformar anillos.
    _ = ax.set_aspect("equal")
    # Ajusta el espacio alrededor de la figura.
    fig.tight_layout()
    # Convierte el gráfico a HTML y lo muestra integrado en el panel.
    display(widgets.HTML(value=imagen_control(fig, titulo)))

In [ ]:
# Crea un control para cambiar gamma y recalcular al soltar el deslizador.
control_kernel = widgets.interactive(
    explorar_kernel,
    gamma=widgets.FloatSlider(value=2.0, min=0.5, max=4, step=0.25, description="Gamma", continuous_update=False),
)
# Presenta el deslizador y el gráfico en un panel interactivo.
display(panel_interactivo("Explora la anchura del kernel", control_kernel))

## Ejercicio 5-2 — Clientes de un mayorista

**Objetivo:** agrupar clientes según sus gastos con los métodos anteriores. La carga está preparada; escribe tus soluciones en las celdas de trabajo.

Las seis columnas son gastos anuales en frescos, leche, comestibles, congelados, detergentes y papel, y delicatessen. No tenemos etiquetas de tipos de cliente.

### 1. Revisar los datos

Ejecuta la carga de `mayorista`. Muestra primeras filas y comprueba dimensiones, tipos de columnas y valores ausentes.

**Consulta:** revisión de medidas en **5.1** y carga con `pd.read_csv` en **5.4.1**.

**Herramientas:** `head()` muestra primeras filas; `shape`, dimensiones; `dtypes`, tipos; `isna()`, valores ausentes.

**Pregunta:** ¿qué representa una fila y qué representa una columna? Conserva `mayorista` con los gastos originales.

In [ ]:
# Lee el archivo de clientes: los campos están separados por tabulaciones.
mayorista = pd.read_csv(DATA_PATH / "Wholesale customers data.csv", sep="\t")
# Comprueba que se han cargado 440 clientes y seis categorías de gasto.
assert mayorista.shape == (440, 6), "Revisa el separador y el archivo de datos."
# Comprueba que las seis columnas se interpretaron como variables numéricas.
assert mayorista.select_dtypes(include="number").shape[1] == 6
# Comprueba que no hay ausentes ni gastos negativos.
assert not mayorista.isna().any().any() and (mayorista >= 0).all().all()

# Apartado 1: revisa filas, dimensiones, tipos y valores ausentes.
# Cada fila representa un cliente y cada columna una categoría de gasto anual.
# Conserva la tabla original para describir los perfiles en sus unidades originales.

### 2. Revisar los gastos y preparar la escala

Compara media, mediana, desviación típica y máximo de cada columna. Después aplica **logaritmo de uno más el gasto** y estandariza. Guarda el resultado como `mayorista_x`, sin modificar `mayorista`.

**Consulta:** `describe()` en **5.1.1** y `escalar` en **5.0.1 y 5.1.2**.

**Paso nuevo:** `np.log1p` calcula $\log(1+x)$. Admite gastos cero y comprime las diferencias entre gastos muy altos. El orden es **logaritmo primero, estandarización después**.

Utiliza `mayorista_x` para agrupar y los gastos originales para interpretar.

**Preguntas:** ¿dónde se separan más media y mediana? ¿Cómo pueden influir los gastos extremos? Comprueba media y desviación típica después de estandarizar.

In [ ]:
# Apartado 2: calcula media, mediana, desviación típica y máximo de cada gasto.
# Después transforma gastos con np.log1p y estandariza el resultado.
# Guarda la tabla transformada como mayorista_x sin cambiar mayorista.
# Comprueba que las columnas estandarizadas tienen media 0 y desviación típica 1.
# Usa los gastos originales al describir los grupos.

### 3. Explorar el número de grupos

Ajusta k-means para $k=2,\ldots,10$ sobre `mayorista_x`, con **Lloyd, diez inicios y `SEED`**. Recoge WSS y Silhouette y dibuja cómo cambian con $k$. Guarda los modelos para reutilizarlos.

**Consulta:** curvas con `ajustar_kmeans` en **5.5.2** y `silhouette_score` en **5.2.6**. WSS está en `inertia_` y las etiquetas en `labels_`. Silhouette recibe los datos del ajuste y esas etiquetas. Puedes usar bloques separados o un bucle.

**Preguntas:** ¿qué $k$ tiene mayor Silhouette? ¿Ves un codo en WSS? ¿Coinciden ambas lecturas?

Después estudiarás de cinco a ocho grupos para practicar una segmentación más detallada, aunque tus índices favorezcan otro número.

In [ ]:
# Apartado 3: ajusta k-means con Lloyd para cada k de 2 a 10.
# Usa mayorista_x, diez inicios y la semilla SEED en cada ajuste.
# Guarda los modelos, la WSS y Silhouette en una tabla.
# Dibuja WSS y Silhouette en gráficos separados para compararlas.
# Comenta qué k favorece Silhouette y dónde parece estar el codo de WSS.

### 4. Comparar formas de iniciar y ajustar k-means

Sobre `mayorista_x`, manteniendo `SEED`, compara:
1. Lloyd con un inicio.
2. Hartigan–Wong con un inicio.
3. Hartigan–Wong con diez inicios.
4. Hartigan–Wong iniciado con los centros de un corte de Ward.

Empieza con **cinco grupos** y repite para **seis, siete y ocho**. Guarda etiquetas y una tabla con WSS, Silhouette y Dunn.

**Consulta:** algoritmo e inicios en **5.5.1** y su control; `init` en **5.0.1**; sistema jerarquía → k-means en **5.5.4**; `linkage` y Ward en **5.2.2–5.2.3**; `cut_tree` en **5.2.7**; `silhouette_score` e `indice_dunn` en **5.2.6**.

**Inicio desde Ward:** construye el árbol con distancias euclídeas de `mayorista_x` y córtalo en $k$ grupos. Selecciona las filas de cada grupo y calcula `mean(axis=0)`: la media de cada columna. Reúne esas medias en una matriz de $k$ filas y seis columnas. Pásala como `init=centros`, con `algoritmo="hartigan-wong"`, `n_init=1` y `random_state=SEED`. El ajuste puede cambiar pertenencias; no necesita ACP.

Calcula los indicadores sobre `mayorista_x`; Dunn necesita sus distancias euclídeas con `squareform(pdist(...))`.

**Preguntas:** para el mismo $k$, ¿más inicios reducen WSS? ¿También mejoran Silhouette? ¿Dunn y Silhouette recomiendan lo mismo? Recuerda: la misma semilla no da los mismos centros en Lloyd y Hartigan–Wong; esta comparación no aísla solo el algoritmo.

In [ ]:
# Apartado 4 — Escribe aquí tu solución.
# Consulta ajustar_kmeans, algoritmo y n_init en el control de 5.5.
# Consulta linkage y Ward en 5.2.2–5.2.3 y cut_tree en 5.2.7.
# Sigue el inicio desde Ward explicado paso a paso en el apartado 4.
# Compara las cuatro opciones para k=5 y repite para 6, 7 y 8.
# Mantén mayorista_x y SEED; guarda cada modelo y sus labels_.
# Selecciona las filas de cada grupo del corte y calcula mean(axis=0).
# Reúne los centros y pásalos mediante init a ajustar_kmeans, con n_init=1.
# Consulta silhouette_score e indice_dunn en 5.2.6.
# Prepara las distancias de Dunn con squareform y pdist, como en 5.2.1.
# Resume WSS, Silhouette y Dunn en una tabla; compara WSS para el mismo k.
# Explica las discrepancias entre indicadores.


### 5. ¿Los métodos forman los mismos grupos?

Compara **Lloyd con un inicio** y **Hartigan–Wong con diez inicios**, para dos grupos y, por separado, para cinco. Usa `mayorista_x` y la semilla **1234** en ambos métodos.

**Consulta:** `ajustar_kmeans` en **5.5.1**; `adjusted_rand_score` en **5.2.9**. Compara los `labels_` de ambos ajustes mediante ARI, sin renombrar las etiquetas. Aquí la referencia es otra partición, no unas clases conocidas.

**Preguntas:** ¿en qué caso es mayor ARI? ¿Son idénticas las particiones? Explica el acuerdo entre métodos, no un porcentaje de clientes bien clasificados.

In [ ]:
# Apartado 5 — Escribe aquí tu solución.
# Consulta ajustar_kmeans en 5.5.1 y adjusted_rand_score en 5.2.9.
# Compara Lloyd con un inicio y Hartigan–Wong con diez.
# Usa mayorista_x y random_state=1234 en ambos métodos.
# Haz la comparación para k=2 y, por separado, para k=5.
# Calcula ARI con sus labels_, sin emparejar los números de grupo.
# Muestra ambos valores y explica en qué caso las particiones se parecen más.


### 6. Describir los grupos de clientes

Recupera el modelo del apartado 4 con **cinco grupos, Hartigan–Wong, diez inicios y `SEED`**.

1. Cuenta clientes por grupo.
2. Calcula las medianas de sus gastos originales.
3. Dibuja los grupos utilizando `mayorista_x`.
4. Describe dos grupos con su tamaño y dos medianas de gasto.

**Consulta:** `np.bincount` en **5.2.7 y 5.5.1**; `representar_clusters` en **5.0.2**; preparación de gastos en el **apartado 2 de este ejercicio**.

**Herramienta nueva:** `groupby` reúne filas con la misma etiqueta; `median()` calcula las medianas de cada grupo. Aplícalas a `mayorista`, no a los datos transformados. Así describes gastos habituales en sus unidades originales.

**Preguntas:** ¿qué grupo destaca en leche y comestibles? ¿Cuál en frescos? Cita cifras sin atribuir tipos de negocio que no conocemos.

**Conclusión:** explica qué aporta estudiar cinco grupos. Distingue compactación, separación, acuerdo entre métodos y descripción del gasto.

In [ ]:
# Apartado 6 — Escribe aquí tu solución.
# Recupera el modelo del apartado 4: k=5, Hartigan–Wong, diez inicios y SEED.
# Consulta np.bincount en 5.2.7 y 5.5.1; representar_clusters en 5.0.2.
# Consulta groupby y median como herramientas nuevas en el apartado 6.
# Cuenta clientes y calcula medianas por grupo sobre mayorista.
# Dibuja con mayorista_x y las etiquetas guardadas.
# Describe dos grupos con tamaños y gastos medianos.
# Justifica qué aporta la segmentación sin atribuir tipos de negocio desconocidos.


## Cierre de la sesión

La escala, las distancias, el método y los inicios pueden cambiar los grupos. Hemos trabajado clustering jerárquico, Hopkins y k-means. En Iris comparamos dos y tres grupos; el disco y la corona muestran una limitación de k-means euclídeo y una alternativa con kernel gaussiano.

**Con tus resultados del mayorista**, resume:
- Qué número de grupos favorecen los indicadores.
- Si cambiar los inicios modifica WSS y pertenencias, incluido el inicio desde Ward.
- Cómo describirías dos grupos mediante tamaños y gastos medianos.

**Consulta:** indicadores en **5.2.6**; inicios en el **apartado 4 del ejercicio 5-2**; ARI en su **apartado 5**; perfiles en su **apartado 6**. Cita una figura y una cifra, sin afirmar que los grupos representan tipos conocidos de cliente.

### Lectura opcional — DBSCAN

DBSCAN agrupa **zonas densas**, sin fijar $k$, y puede dejar puntos fuera de los grupos. No calcula centroides como k-means.

- **`eps` ($\varepsilon$):** radio para buscar vecinos, no diámetro máximo de un grupo.
- **`min_samples`:** mínimo de vecinos en ese radio, contando el propio punto.
- **Núcleo:** alcanza ese mínimo y permite extender el grupo a otros núcleos conectados.
- **Frontera:** no alcanza el mínimo, pero está cerca de un núcleo; se incorpora sin extender el grupo.
- **Ruido:** queda fuera y recibe **−1**, no un grupo adicional ni necesariamente un error real.

Un radio pequeño puede fragmentar grupos; uno grande puede unirlos. La escala importa y un solo radio puede fallar con densidades muy diferentes. DBSCAN puede reconocer curvas densas y dejar puntos aislados como ruido; **no equivale a k-means de kernel**, que utiliza otro criterio de agrupación.

Esta lectura completa el recorrido conceptual, sin código ni tareas adicionales. Para recordar por qué la forma y la escala importan, consulta **5.1.2 y 5.6**.

### Documentación

Referencias para interpretar los algoritmos y los índices; no añaden dependencias al cuaderno:
- [Clustering jerárquico y Ward en SciPy](https://docs.scipy.org/doc/scipy/reference/cluster.hierarchy.html).
- [K-means en scikit-learn (Lloyd)](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html).
- [K-means y Hartigan–Wong en el manual de R](https://stat.ethz.ch/R-manual/R-devel/library/stats/html/kmeans.html); `ajustar_kmeans` ofrece también este algoritmo mediante el módulo auxiliar del proyecto.
- [Kernel gaussiano RBF: similitud, gamma y distancia](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.rbf_kernel.html).
- [Índice de Rand ajustado (ARI)](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html).
- [Silhouette: cohesión y separación](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html).
- [Selección del número de grupos mediante indicadores internos](https://www.jstatsoft.org/article/view/v061i06).